In [17]:
import pandas as pd
import numpy as np


train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
submission = pd.read_csv('sampleSubmission.csv')

In [18]:
train = train[train['weather'] != 4]

In [19]:
print(train.shape[0])
print(train.shape[1])
print(test.shape[0])
print(test.shape[1])

10885
12
6493
9


In [20]:
df = pd.concat([train,test], ignore_index=True)

df.shape[0]

df


,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0000,3.0,13.0,16.0
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0000,8.0,32.0,40.0
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0000,5.0,27.0,32.0
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0000,3.0,10.0,13.0
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0000,0.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
17373,2012-12-31 19:00:00,1,0,1,2,10.66,12.880,60,11.0014,NaN,NaN,NaN
17374,2012-12-31 20:00:00,1,0,1,2,10.66,12.880,60,11.0014,NaN,NaN,NaN
17375,2012-12-31 21:00:00,1,0,1,1,10.66,12.880,60,11.0014,NaN,NaN,NaN
17376,2012-12-31 22:00:00,1,0,1,1,10.66,13.635,56,8.9981,NaN,NaN,NaN


In [21]:
from datetime import datetime

df['date'] = df['datetime'].apply(lambda x : x.split()[0]) # date
df['year'] = df['datetime'].apply(lambda x : x.split()[0].split('-')[0]) #year
df['month'] = df['datetime'].apply(lambda x : x.split()[0].split('-')[1])
#month

df['hour'] = df['datetime'].apply(lambda x : x.split()[1].split(':')[0])#hour

df['day'] = df['date'].apply(lambda x : datetime.strptime(x,'%Y-%m-%d').weekday())

drop_cols = ['casual','registered','datetime','date','windspeed','month']
df = df.drop(drop_cols, axis=1)

In [22]:
# Split train - test dataset again after feature engineering

X_train = df[~pd.isnull(df['count'])]
X_test = df[pd.isnull(df['count'])] # ㅅㅂ 천재냐?

X_train = X_train.drop(['count'], axis=1)
X_test = X_test.drop(['count'], axis=1)

y = train['count']

X_train.head()

,season,holiday,workingday,weather,temp,atemp,humidity,year,hour,day
0,1,0,0,1,9.84,14.395,81,2011,00,5
1,1,0,0,1,9.02,13.635,80,2011,01,5
2,1,0,0,1,9.02,13.635,80,2011,02,5
3,1,0,0,1,9.84,14.395,75,2011,03,5
4,1,0,0,1,9.84,14.395,75,2011,04,5


In [23]:
## Construct Evaluation Matrix - RMSLE

import numpy as np

def rmsle(y_true, y_pred, convertExp = True):
    if convertExp:
        y_true = np.exp(y_true)
        y_pred = np.exp(y_pred)
        
    log_true =np.nan_to_num(np.log(y_true+1))
    log_pred =np.nan_to_num(np.log(y_pred+1))
    
    output = np.sqrt(np.mean((log_true-log_pred)**2))
    return output

In [24]:
from sklearn.linear_model import LinearRegression

linear_reg = LinearRegression()

log_y = np.log(y)
linear_reg.fit(X_train, log_y)

pred = linear_reg.predict(X_train)

print(rmsle(log_y, pred))

1.0204980189305088


In [25]:
linear_reg_pred = linear_reg.predict(X_test)

submission['count']  = np.exp(linear_reg_pred)

submission.to_csv('submission.csv',index=False)
